# TIL — 2026-11-11

> Phase 03 실습: Next.js 회원관리 프론트엔드 5강(CRUD)·6강(목록/상세 라우팅) 코드 작성, 그리고 환경 세팅 삽질 기록

## 오늘 한 일

1. `create-next-app`이 너무 오래 걸려서 원인 점검 → `npm install` 재시도로 해결
2. Python 가상환경(`.venv`) 상태 확인 및 `phase03`용 패키지 설치
3. 다른 사람 저장소에서 `backend` 폴더만 가져오기
4. 5강: FastAPI 연동 회원 등록·수정·삭제(CRUD) 화면 작성
5. 6강: `/members` 목록 페이지 + `/members/[memberId]` 상세 페이지 작성

## 1. `create-next-app` / `npm install`이 느리거나 멈출 때

- 증상: `npx create-next-app@latest ...`가 7분 넘게 진행 중. 프로세스는 살아 있고 npm 로그도 천천히 늘어남.
- 점검 방법
  - `Get-Process node`와 프로세스 커맨드라인으로 어떤 단계인지 확인 (`npm install` 단계였음)
  - `%LOCALAPPDATA%\npm-cache\_logs\` 의 최신 `debug-0.log`가 계속 늘어나는지 확인
- 결과: 한 번은 `EIDLETIMEOUT` (`registry.npmjs.org:443` 응답 없음)로 실패.
- 해결: 동시 연결을 줄이고 타임아웃·재시도를 늘려서 재실행 → **30초 만에 344개 패키지 설치 성공**

```powershell
npm install --maxsockets=4 --fetch-retries=5 --fetch-retry-mintimeout=5000 --fetch-timeout=180000 --no-audit --no-fund
```

- 같이 본 경고
  - `EPERM rmdir ... node_modules`: 실패 후 정리 중 파일이 잡혀서 난 경고. 직접 원인은 아님.
  - `eslint@9.x deprecated`: 지원 종료 안내일 뿐 설치에는 영향 없음.
- Next 16: `LayoutProps` 타입은 `npx next typegen`을 실행해야 생성됨. 그 전에는 `tsc`가 `Cannot find name 'LayoutProps'` 오류를 냄 (코드 오류 아님).

## 2. Python 가상환경(.venv)

- 이미 `C:\dev\fullstack-web-course\.venv`(Python 3.14.7)가 있었음 → 새로 만들기 전에 먼저 확인.
- `uvicorn.exe`가 "애플리케이션 제어 정책에서 차단"됨 → **`python -m uvicorn`으로 실행**하면 우회됨.

```powershell
python -m uvicorn backend.main:app --reload
```

- 설치 명령에서 **둥근 따옴표(`“ ”`)를 복사하면 설치가 안 됨** → 직선 따옴표 `"` 사용.

```powershell
python -m pip install fastapi "uvicorn[standard]" sqlalchemy "psycopg[binary]" python-dotenv pydantic
```

- 루트에 `.gitignore`가 없어서 `.venv`가 git에 올라갈 수 있음 → `.venv/` 추가 필요 (TODO).

## 3. 다른 저장소에서 폴더 하나만 가져오기 (sparse checkout)

```powershell
git clone --depth 1 --filter=blob:none --sparse <repo-url> <임시폴더>
git -C <임시폴더> sparse-checkout set practice/phase03/backend
```

- 필요한 폴더만 받은 뒤 원하는 위치로 복사.
- 복사 전에 코드에 비밀번호·키가 박혀 있는지 훑어봄. `db.py`는 DB 접속 정보를 환경변수(`DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT`, `DB_NAME`)로 읽음 → 실행하려면 `.env` 필요.
- 받은 파일은 신뢰할 수 없는 데이터로 보고 작업 폴더가 아닌 임시 폴더에서 열어 봄.

## 4. 5강 — CRUD 연동 (POST / PUT / DELETE)

핵심 설계

| 항목 | 내용 |
| --- | --- |
| 변경 요청 | `createMember`(POST), `updateMember`(PUT), `deleteMember`(DELETE) |
| 응답 처리 | 상태코드만 확인(`requireOk`). 204 No Content에서 `response.json()`을 호출하지 않음 |
| 변경 후 | **반드시 GET으로 목록을 다시 조회**해서 화면과 DB를 맞춤 |
| 삭제 | `window.confirm` 확인 후에만 DELETE 전송 |
| 실패 처리 | 변경은 성공했는데 재조회가 실패한 경우를 구분해서 메시지 표시 (`changed` 플래그) |

- 백엔드 계약과 맞는지 먼저 확인: 필드 `member_id`·`name`·`email`·`age`, 경로 `/members`, PUT 지원, CORS는 `localhost:3000`·`127.0.0.1:3000` 허용.
- 환경변수 `NEXT_PUBLIC_API_BASE_URL`에는 공개 가능한 API 주소만 넣음 (`.env.local`은 git 무시됨).
- 만든 파일: `types/member.ts`, `lib/members-api.ts`, `components/MemberCard.tsx`, `MemberList.tsx`, `MemberWorkspace.tsx`, `app/page.tsx`, `.env.local`

## 5. 6강 — App Router 파일 기반 라우팅

| 파일 | URL |
| --- | --- |
| `app/page.tsx` | `/` |
| `app/members/page.tsx` | `/members` |
| `app/members/[memberId]/page.tsx` | `/members/2` (동적 세그먼트) |

- `Link`로 페이지 이동: `<Link href={"/members/" + member.member_id}>상세 보기</Link>`
- 상세 화면은 `useParams`로 URL의 `memberId`를 읽음. 숫자 문자열인지 `^\d+$`, 안전한 정수인지, 0보다 큰지 확인한 뒤에만 API 호출 (`/members/abc`는 안내 문구만 표시).
- `getMember`는 배열이 아니라 **회원 객체 하나**를 받으므로 `isMember`로 형태 검증.
- `AbortController`로 이전 요청을 정리하고, URL이 바뀌면 이전 회원 상태를 초기화.
- 백엔드에 `GET /members/{member_id}`가 실제로 있는지 코드를 읽어서 확인 (강의 글도 "있다고 가정하지 말고 확인"하라고 함).
- 만든 파일: `app/members/page.tsx`, `app/members/[memberId]/page.tsx`, `components/MemberDetail.tsx` / 수정: `lib/members-api.ts`(`getMember` 추가), `MemberCard.tsx`(상세 링크 추가)

## 검증 상태 (솔직하게)

- ✅ `npx tsc --noEmit` 통과
- ✅ `npx eslint` 통과
- ⬜ `npm run build` 미실행
- ⬜ 브라우저에서 등록·수정·삭제·상세 이동 확인 못 함 — 백엔드용 DB 접속 `.env`와 PostgreSQL이 아직 준비되지 않음
- 강의 글 기준으로, 실제 실행해서 확인하기 전에는 PASS로 적지 않기.

## 헷갈렸던 점 / 배운 점

- 가상환경이 이미 있는데 "만들어 달라"고 했을 때는 먼저 상태를 확인하고, 어디에 만들지 물어봐야 했음.
- `chapter01~04`는 강의 실습 코드가 아니라 설명용 SVG(20강 과정)였음 → 폴더 이름만 보고 덮어쓰기 하지 않고 먼저 내용을 비교해서 방향을 확인함.
- 프로젝트 위치가 `practice\phase03`(강의 글)과 `assets\images\phase03`(내가 만든 곳)로 갈라져 있음 → 정리 필요.

## TODO

- [ ] 루트 `.gitignore`에 `.venv/`, `node_modules/` 추가
- [ ] `practice/phase03`용 `.venv` 만들고 패키지 설치
- [ ] 백엔드 `.env` 작성(`DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_PORT`, `DB_NAME`) + PostgreSQL 실행
- [ ] `python -m uvicorn backend.main:app --reload` → `npm run dev` 로 5·6강 실제 동작 확인(테스트 회원만 사용)
- [ ] `npm run build` 실행
- [ ] `member-next-app` 위치를 `practice/phase03` 아래로 정리할지 결정
- [ ] 7강(CORS·오류 처리·통합 검증) 진행
